In [ ]:
import logging
from IPython.display import Markdown
from library.circuitry import Circuitry
from library.junction_patch import JunctionPatch
from library.qubit_array import QubitArray
from library.surface_code.expanding_patch import ExpandingSurfaceCodePatch
from library.surface_code.patch import SurfaceCodePatch
from library.common import Pauli
from library.steane_code.patch import SteaneCodePatch
from utils.simulation.stim import simulate, sample

logging.basicConfig(level=logging.ERROR)

In [ ]:
def detector_report(circuitry: Circuitry) -> str:
    issues = []
    if len(circuitry.missing_detectors()) > 0:
        issues.append("MISSING")

    try:
        circuitry.as_stim.detector_error_model(allow_gauge_detectors=False)
    except ValueError:
        issues.append("GAUGE")

    return "&".join(issues)

In [ ]:
TARGET_DISTANCE = 9

SUPERDENSE_ROUNDS = 3
TELEPORT_ROUNDS = 3
ROUNDS_FOR_COMPLEMENTARY_GAP = 1
EXPANDED_OPACITY = 0.125

if TARGET_DISTANCE % 2 != 1 and TARGET_DISTANCE < 9:
    raise ValueError("TARGET_DISTANCE must be odd and above 9.")

In [ ]:
FILEROOT = "generated/hirano-magic-state-cultivation-layout1"
scenarios: dict[str, Circuitry] = dict()
point = 0

In [ ]:
# Generate circuit up to and including preparation with S-injection
qubits = QubitArray(dimensions=(TARGET_DISTANCE + 2, TARGET_DISTANCE + 2))
circuitry = Circuitry(qubits, clifford=True)
steane = SteaneCodePatch(
    qubits,
    anchor=(TARGET_DISTANCE - 5, TARGET_DISTANCE - 7),
    injection=SteaneCodePatch.Injection.S,
)

circuitry.annotate_polygons(steane.get_polygons(initial=True))

# Append preparation
steane.append_preparation(circuitry)

circuitry.annotate_polygons(steane.get_polygons(initial=False))

circuitry.append_observable(0, "Y_OBSERVABLE_PREPARED", steane.logical(Pauli.Y))

scenarios["Prepared"] = circuitry
circuitry.to_file(FILEROOT + f".point{point}.prepared")
point += 1

In [ ]:
# Generate circuit up to and including superdense code cycles
qubits = QubitArray(dimensions=(TARGET_DISTANCE + 2, TARGET_DISTANCE + 2))
circuitry = Circuitry(qubits, clifford=True)
steane = SteaneCodePatch(
    qubits,
    anchor=(TARGET_DISTANCE - 5, TARGET_DISTANCE - 7),
    injection=SteaneCodePatch.Injection.S,
)

circuitry.annotate_polygons(steane.get_polygons(initial=True))

steane.append_preparation(circuitry)

circuitry.annotate_polygons(steane.get_polygons(initial=False))

for s in range(SUPERDENSE_ROUNDS):
    label = f"SDC{s}"
    steane.append_superdense_cycle(circuitry, prefix=label)

steane.annotate_detectors(circuitry, sdc_rounds=SUPERDENSE_ROUNDS, tpt_rounds=0)
circuitry.append_observable(0, "Y_OBSERVABLE_SUPERDENSED", steane.logical(Pauli.Y))

scenarios[f"SDCx{SUPERDENSE_ROUNDS}"] = circuitry
circuitry.to_file(FILEROOT + f".point{point}.superdense")
point += 1

In [ ]:
# Generate circuit up to and including double-check-S
qubits = QubitArray(dimensions=(TARGET_DISTANCE + 2, TARGET_DISTANCE + 2))
circuitry = Circuitry(qubits, clifford=True)
steane = SteaneCodePatch(
    qubits,
    anchor=(TARGET_DISTANCE - 5, TARGET_DISTANCE - 7),
    injection=SteaneCodePatch.Injection.S,
)

circuitry.annotate_polygons(steane.get_polygons(initial=True))

steane.append_preparation(circuitry)

circuitry.annotate_polygons(steane.get_polygons(initial=False))

for s in range(SUPERDENSE_ROUNDS):
    label = f"SDC{s}"
    steane.append_superdense_cycle(circuitry, prefix=label)
steane.append_cultivation(circuitry, prefix="CULT")

steane.annotate_detectors(circuitry, sdc_rounds=SUPERDENSE_ROUNDS)
circuitry.append_observable(0, "Y_OBSERVABLE_DOUBLE_CHECKED", steane.logical(Pauli.Y))

scenarios["Double-Check-S"] = circuitry
circuitry.to_file(FILEROOT + f".point{point}.double-check-s")
point += 1

In [ ]:
def inactive_source(location: tuple[float, float]) -> bool:
    return location[1] == TARGET_DISTANCE - 4.5


# Generate circuit up to and including teleportation
qubits = QubitArray(dimensions=(TARGET_DISTANCE + 2, TARGET_DISTANCE + 2))
circuitry = Circuitry(qubits, clifford=True)
steane = SteaneCodePatch(
    qubits,
    anchor=(TARGET_DISTANCE - 5, TARGET_DISTANCE - 7),
    injection=SteaneCodePatch.Injection.S,
)
junction = JunctionPatch(qubits, anchor=(TARGET_DISTANCE - 5, TARGET_DISTANCE - 5))
source = SurfaceCodePatch(
    qubits, distance=5, anchor=(TARGET_DISTANCE - 4, TARGET_DISTANCE - 4)
)

circuitry.annotate_polygons(steane.get_polygons(initial=True))

steane.append_preparation(circuitry)

circuitry.annotate_polygons(steane.get_polygons(initial=False))

for s in range(SUPERDENSE_ROUNDS):
    label = f"SDC{s}"
    steane.append_superdense_cycle(circuitry, prefix=label)
steane.append_cultivation(circuitry, prefix="CULT")

circuitry.annotate_polygons(steane.get_polygons())
circuitry.annotate_polygons(junction.get_polygons())
circuitry.annotate_polygons(source.get_polygons(inactive_source))

for rnd in range(TELEPORT_ROUNDS):
    for mmt in steane.TELEPORTATION_MOMENTS:
        steane.append_teleportation(circuitry, moment=mmt, prefix=f"TPT{rnd}")
        junction.append_syndrome(circuitry, moment=mmt, prefix=f"JCT{rnd}")
        source.append_round(
            circuitry,
            moment=mmt,
            prepare=Pauli.X if rnd == 0 else None,
            prefix=f"SC{rnd}",
            inactive=inactive_source,
        )
        circuitry.append_tick()

circuitry.annotate_polygons(steane.get_polygons(opacity=2.25 * EXPANDED_OPACITY))
circuitry.annotate_polygons(source.get_polygons())

for mmt in steane.DESTRUCTION_MOMENTS:
    steane.append_destruction(circuitry, moment=mmt)
    source.append_round(circuitry, moment=mmt, prefix=f"SC{TELEPORT_ROUNDS}")
    circuitry.append_tick()

circuitry.annotate_polygons(source.get_polygons())

circuitry.append_observable(
    0,
    "Y_OBSERVABLE_TELEPORTED",
    source.logical(Pauli.Y),
    "JCT0:Z0",
    "JCT0:Z1",
    "JCT0:Z2",
    "TPT0:XB",
    "TPT1:XB",
    "TPT2:XB",
    "DST:X1",
    "DST:X5",
    "DST:X6",
)

steane.annotate_detectors(
    circuitry, sdc_rounds=SUPERDENSE_ROUNDS, tpt_rounds=TELEPORT_ROUNDS
)
junction.annotate_detectors(circuitry, rounds=TELEPORT_ROUNDS)
source.annotate_detectors(circuitry, rounds=TELEPORT_ROUNDS + 1, prepared=Pauli.X)

scenarios["Teleported"] = circuitry
circuitry.to_file(FILEROOT + f".point{point}.teleported")
point += 1

In [ ]:
# Generate the circuit up to and including expansion :)
qubits = QubitArray(dimensions=(TARGET_DISTANCE + 2, TARGET_DISTANCE + 2))
circuitry = Circuitry(qubits, clifford=True)
steane = SteaneCodePatch(
    qubits,
    anchor=(TARGET_DISTANCE - 5, TARGET_DISTANCE - 7),
    injection=SteaneCodePatch.Injection.S,
)
junction = JunctionPatch(qubits, anchor=(TARGET_DISTANCE - 5, TARGET_DISTANCE - 5))
source = SurfaceCodePatch(
    qubits, distance=5, anchor=(TARGET_DISTANCE - 4, TARGET_DISTANCE - 4)
)
target = ExpandingSurfaceCodePatch(
    qubits, distance=5, anchor=(1, 1), expansion=TARGET_DISTANCE - 5
)

circuitry.annotate_polygons(
    target.get_polygons(expanded=True, opacity=EXPANDED_OPACITY)
)
circuitry.annotate_polygons(steane.get_polygons(initial=True))

steane.append_preparation(circuitry)

circuitry.annotate_polygons(
    target.get_polygons(expanded=True, opacity=EXPANDED_OPACITY)
)
circuitry.annotate_polygons(steane.get_polygons(initial=False))

for s in range(SUPERDENSE_ROUNDS):
    label = f"SDC{s}"
    steane.append_superdense_cycle(circuitry, prefix=label)
steane.append_cultivation(circuitry, prefix="CULT")

circuitry.annotate_polygons(
    target.get_polygons(expanded=True, opacity=EXPANDED_OPACITY)
)
circuitry.annotate_polygons(steane.get_polygons())
circuitry.annotate_polygons(junction.get_polygons())
circuitry.annotate_polygons(source.get_polygons(inactive_source))

for rnd in range(TELEPORT_ROUNDS):
    for mmt in steane.TELEPORTATION_MOMENTS:
        steane.append_teleportation(circuitry, moment=mmt, prefix=f"TPT{rnd}")
        junction.append_syndrome(circuitry, moment=mmt, prefix=f"JCT{rnd}")
        source.append_round(
            circuitry,
            moment=mmt,
            prepare=Pauli.X if rnd == 0 else None,
            prefix=f"SC{rnd}",
            inactive=inactive_source,
        )
        circuitry.append_tick()

circuitry.annotate_polygons(
    target.get_polygons(expanded=True, opacity=EXPANDED_OPACITY)
)
circuitry.annotate_polygons(steane.get_polygons(opacity=2.25 * EXPANDED_OPACITY))
circuitry.annotate_polygons(source.get_polygons())

for mmt in steane.DESTRUCTION_MOMENTS:
    steane.append_destruction(circuitry, moment=mmt)
    source.append_round(circuitry, moment=mmt, prefix=f"SC{TELEPORT_ROUNDS}")
    circuitry.append_tick()

circuitry.annotate_polygons(target.get_polygons(expanded=True))
for mmt in target.MOMENTS:
    target.append_expansion(circuitry, moment=mmt, prefix="EXP")
    circuitry.append_tick()

circuitry.append_observable(
    0,
    "Y_OBSERVABLE_EXPANDED",
    target.logical(Pauli.Y),
    "JCT0:Z0",
    "JCT0:Z1",
    "JCT0:Z2",
    "TPT0:XB",
    "TPT1:XB",
    "TPT2:XB",
    "DST:X1",
    "DST:X5",
    "DST:X6",
)

steane.annotate_detectors(
    circuitry, sdc_rounds=SUPERDENSE_ROUNDS, tpt_rounds=TELEPORT_ROUNDS
)
junction.annotate_detectors(circuitry, rounds=TELEPORT_ROUNDS)
source.annotate_detectors(circuitry, rounds=TELEPORT_ROUNDS + 1, prepared=Pauli.X)
target.annotate_detectors(circuitry, sc_rounds=TELEPORT_ROUNDS + 1, source=source)

scenarios["Expanded"] = circuitry
circuitry.to_file(FILEROOT + f".point{point}.expanded")
point += 1

In [ ]:
for index, (scenario, circuitry) in enumerate(scenarios.items()):
    warning = detector_report(circuitry)
    display(
        Markdown(
            f"[Open in Crumble (Point {index} - {scenario})]({circuitry.to_crumble_url()}) {warning}"
        )
    )

In [ ]:
# Analyse error rates of all cumulative circuits
title = r"Magic State Cultivation of $|\mathbf{S}\rangle$ [Corrected $\overline{\mathbf{Y}}$]"
sample(scenarios, title=title, label="Point", shots=1e6, correction=True, fontsize=10)

In [ ]:
simulate(
    scenarios,
    title,
    label="Point",
    postselection=True,
    shots=1e6,
    minimal_noise=-6,
    figsize=(11, 4.5),
    num_workers=7,
)